# USGS handcrafted terrain baseline

Import the libraries for reading DEMs and fitting small classifiers. This notebook tests whether simple elevation features predict facility size on the same spatial folds as the terrain CNN.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import rasterio

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

**Check the kernel environment**

Print the Python executable, working directory, and whether the Windows mount is visible. This confirms the notebook is running in the expected environment before any data path is trusted.

In [3]:
from pathlib import Path
import os
import sys

print("Python executable:", sys.executable)
print("Current directory:", Path.cwd())
print("Home directory:", Path.home())
print("Is /mnt/c available?", Path("/mnt/c").exists())

print("\nCurrent directory contents:")
for p in sorted(Path.cwd().iterdir()):
    print(" ", p.name)

Python executable: /home/atthota/venvs/atlas/bin/python
Current directory: /mnt/c/Users/attho/OneDrive/Desktop/College/ACM ATLAS/ATLAS/models
Home directory: /home/atthota
Is /mnt/c available? True

Current directory contents:
  USGS_CNN.ipynb
  USGS_CNN_complete.ipynb
  USGS_Terrain_Baseline.ipynb
  data
  local_training_eval.ipynb
  model_training_eval.ipynb
  outputs
  tabular_baseline
  train_multimodal.ipynb
  train_vision_baseline.py


**Search for folds and CNN predictions**

Resolve the repository root and list fold files, then check that the CNN out-of-fold predictions exist. The printed row and fold counts show whether that experiment finished.

In [5]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.resolve()

print("Project root:", PROJECT_ROOT)

print("\nSearching for saved spatial folds:")
for path in PROJECT_ROOT.rglob("*fold*.csv"):
    print(path.relative_to(PROJECT_ROOT))

print("\nChecking CNN predictions:")
cnn_path = (
    PROJECT_ROOT
    / "outputs/usgs_cnn_spatial_v1_30epochs/oof_predictions.csv"
)

print("CNN predictions exist:", cnn_path.exists())

if cnn_path.exists():
    import pandas as pd

    cnn_check = pd.read_csv(cnn_path, dtype={"id": str})
    print("Rows:", len(cnn_check))
    print("Unique facilities:", cnn_check["id"].nunique())
    print("Folds:", sorted(cnn_check["fold"].unique()))

Project root: /mnt/c/Users/attho/OneDrive/Desktop/College/ACM ATLAS/ATLAS

Searching for saved spatial folds:
outputs/usgs_cnn_spatial_v1/fold_metrics.csv
outputs/usgs_cnn_spatial_v1_30epochs/fold_metrics.csv
outputs/tabular_baselines/atlas_wri_v3/explainability/wri_ablation_fold_results.csv
outputs/tabular_baselines/atlas_wri_v2/geographic_cv/geographic_fold_manifest.csv
outputs/tabular_baselines/atlas_wri_v2/geographic_cv/rf_baseline_folds.csv
outputs/tabular_baselines/atlas_wri_v2/geographic_cv/unweighted_xgb_folds.csv
outputs/tabular_baselines/atlas_wri_v2/geographic_cv/weighted_catboost_folds.csv
outputs/tabular_baselines/atlas_wri_v2/geographic_cv/weighted_xgb_folds.csv
outputs/tabular_baselines/atlas_wri_v2/geographic_cv/xgboost_tuning/candidate_fold_metrics.csv
models/data/processed/usgs_naip_alignment/usgs_spatial_folds_v1.csv

Checking CNN predictions:
CNN predictions exist: True
Rows: 180
Unique facilities: 180
Folds: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.i

**Set input and output paths**

Point at the DEM directory, spatial folds, facility registry, CNN outputs, and the handcrafted output folder. Required paths are checked before feature extraction starts.

In [6]:
from pathlib import Path

# Notebook location: ATLAS/models/
PROJECT_ROOT = Path.cwd().parent.resolve()
MODELS_DIR = PROJECT_ROOT / "models"

# USGS data
DEM_DIR = PROJECT_ROOT / "data/temp/usgs_naip_aligned_v2"

# Original spatial folds are stored under models/data/
FOLDS_PATH = (
    MODELS_DIR
    / "data/processed/usgs_naip_alignment/usgs_spatial_folds_v1.csv"
)

# Facility registry
REGISTRY_PATH = PROJECT_ROOT / "data/processed/im3_facility_registry.csv"

# Completed CNN experiment
CNN_DIR = PROJECT_ROOT / "outputs/usgs_cnn_spatial_v1_30epochs"

# Handcrafted baseline outputs
OUT_DIR = PROJECT_ROOT / "outputs/usgs_handcrafted_spatial_v1"

# Verify required inputs
required_paths = {
    "DEM directory": DEM_DIR,
    "Spatial folds": FOLDS_PATH,
    "Facility registry": REGISTRY_PATH,
    "CNN predictions": CNN_DIR / "oof_predictions.csv",
}

print("Checking input paths:\n")

for name, path in required_paths.items():
    print(f"{name}: {path}")
    print(f"  Exists: {path.exists()}")

missing = [
    name for name, path in required_paths.items()
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        f"Missing required inputs: {missing}"
    )

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("\nAll input paths verified.")
print("Output directory:", OUT_DIR)

Checking input paths:

DEM directory: /mnt/c/Users/attho/OneDrive/Desktop/College/ACM ATLAS/ATLAS/data/temp/usgs_naip_aligned_v2
  Exists: True
Spatial folds: /mnt/c/Users/attho/OneDrive/Desktop/College/ACM ATLAS/ATLAS/models/data/processed/usgs_naip_alignment/usgs_spatial_folds_v1.csv
  Exists: True
Facility registry: /mnt/c/Users/attho/OneDrive/Desktop/College/ACM ATLAS/ATLAS/data/processed/im3_facility_registry.csv
  Exists: True
CNN predictions: /mnt/c/Users/attho/OneDrive/Desktop/College/ACM ATLAS/ATLAS/outputs/usgs_cnn_spatial_v1_30epochs/oof_predictions.csv
  Exists: True

All input paths verified.
Output directory: /mnt/c/Users/attho/OneDrive/Desktop/College/ACM ATLAS/ATLAS/outputs/usgs_handcrafted_spatial_v1


**Load CNN predictions, folds, and the registry**

Read the CNN out-of-fold file, the saved spatial folds, and the facility registry, and align ids to 10 digits. Basic uniqueness checks confirm the 180-facility cohort before features are built.

In [7]:
cnn_oof = pd.read_csv(
    CNN_DIR / "oof_predictions.csv",
    dtype={"id": str}
)

cnn_oof["id"] = cnn_oof["id"].str.zfill(10)

folds = pd.read_csv(
    FOLDS_PATH,
    dtype={"id": str}
)

folds["id"] = folds["id"].str.zfill(10)

registry = pd.read_csv(
    REGISTRY_PATH,
    dtype={"id": str}
)

registry["id"] = registry["id"].str.zfill(10)

print("CNN predictions:", len(cnn_oof))
print("Fold rows:", len(folds))
print("Registry rows:", len(registry))
print("Fold columns:", folds.columns.tolist())

assert len(cnn_oof) == 180
assert cnn_oof["id"].is_unique
assert folds["id"].is_unique
assert registry["id"].is_unique
assert set(cnn_oof["id"]).issubset(set(folds["id"]))

display(cnn_oof.head())

CNN predictions: 180
Fold rows: 180
Registry rows: 1240
Fold columns: ['id', 'lat', 'lon', 'size_class', 'geo_group', 'fold']


,id,y_true,y_pred,prob_small,prob_medium,prob_large,fold
0,0014930068,2,2,0.325290,0.329494,0.345216,1
1,0014997588,2,2,0.322523,0.325319,0.352157,1
2,0016282459,1,2,0.323914,0.335700,0.340387,1
3,0031800424,1,2,0.322088,0.325023,0.352889,1
4,0032560290,2,2,0.322273,0.323674,0.354053,1


**Build the evaluation cohort**

Keep id, label, and fold from the CNN predictions and attach square footage from the registry. Labels are checked against the fixed footprint thresholds so this baseline scores the same target as the CNN.

In [8]:
df = cnn_oof[["id", "y_true", "fold"]].copy()

df = df.merge(
    registry[["id", "sqft"]],
    on="id",
    how="left",
    validate="one_to_one"
)

assert df["sqft"].notna().all()
assert df["y_true"].isin([0, 1, 2]).all()
assert df["fold"].isin([1, 2, 3, 4, 5]).all()

# Verify labels agree with the established ATLAS thresholds.
expected_labels = np.select(
    [
        df["sqft"] < 20_000,
        df["sqft"] < 100_000,
    ],
    [0, 1],
    default=2,
)

assert np.array_equal(
    df["y_true"].to_numpy(),
    expected_labels
), "Saved CNN labels disagree with registry thresholds."

print("Facilities:", len(df))
print("Class counts:")
print(df["y_true"].value_counts().sort_index())

Facilities: 180
Class counts:
y_true
0      7
1     50
2    123
Name: count, dtype: int64


**Define terrain feature extraction**

Summarize each DEM with elevation and slope statistics on valid pixels, using approximate meter spacing. These handcrafted numbers are the only inputs the models below will see.

In [9]:
def extract_terrain_features(dem_path):
    with rasterio.open(dem_path) as src:
        dem = src.read(1, masked=True).astype(np.float64)
        transform = src.transform

    valid_mask = ~np.ma.getmaskarray(dem)
    valid = np.asarray(dem.data)[valid_mask]

    if valid.size < 20:
        raise ValueError(f"Too few valid pixels: {dem_path}")

    center = np.median(valid)
    centered = np.asarray(dem.filled(center)) - center

    # Approximate meter spacing for geographic CRS (EPSG:4326).
    lat_center = transform.f + transform.e * (dem.shape[0] / 2)

    meters_per_degree_lat = 111_132.0
    meters_per_degree_lon = (
        111_320.0 * np.cos(np.deg2rad(lat_center))
    )

    dx = abs(transform.a) * meters_per_degree_lon
    dy = abs(transform.e) * meters_per_degree_lat

    if dx <= 0 or dy <= 0:
        raise ValueError(f"Invalid DEM pixel spacing: {dem_path}")

    # Only gradients with valid neighboring pixels are retained.
    gx = (centered[:, 1:] - centered[:, :-1]) / dx
    gy = (centered[1:, :] - centered[:-1, :]) / dy

    gx_valid = valid_mask[:, 1:] & valid_mask[:, :-1]
    gy_valid = valid_mask[1:, :] & valid_mask[:-1, :]

    gx_values = gx[gx_valid]
    gy_values = gy[gy_valid]

    if gx_values.size == 0 or gy_values.size == 0:
        raise ValueError(f"No valid terrain gradients: {dem_path}")

    # Approximate gradient magnitudes using aligned interior differences.
    interior_gx = (
        centered[1:, 1:] - centered[1:, :-1]
    ) / dx

    interior_gy = (
        centered[1:, 1:] - centered[:-1, 1:]
    ) / dy

    interior_valid = (
        valid_mask[1:, 1:]
        & valid_mask[1:, :-1]
        & valid_mask[:-1, 1:]
    )

    gradient_magnitude = np.sqrt(
        interior_gx**2 + interior_gy**2
    )[interior_valid]

    if gradient_magnitude.size == 0:
        raise ValueError(f"No valid gradient magnitudes: {dem_path}")

    return {
        "elev_std_m": float(np.std(valid)),
        "elev_range_m": float(np.ptp(valid)),
        "elev_iqr_m": float(
            np.percentile(valid, 75) - np.percentile(valid, 25)
        ),
        "elev_p10_relative_m": float(
            np.percentile(valid, 10) - center
        ),
        "elev_p90_relative_m": float(
            np.percentile(valid, 90) - center
        ),
        "gradient_mean": float(np.mean(gradient_magnitude)),
        "gradient_std": float(np.std(gradient_magnitude)),
        "gradient_p90": float(np.percentile(gradient_magnitude, 90)),
        "roughness_x_m": float(np.mean(np.abs(gx_values)) * dx),
        "roughness_y_m": float(np.mean(np.abs(gy_values)) * dy),
    }

**Extract features for every DEM**

Run that summary on all 180 facility DEMs and save the feature table. A missing tile stops the loop so the cohort cannot silently shrink.

In [10]:
records = []

for i, row in enumerate(df.itertuples(index=False), start=1):
    dem_path = DEM_DIR / f"{row.id}_dem.tif"

    if not dem_path.exists():
        raise FileNotFoundError(dem_path)

    features = extract_terrain_features(dem_path)

    records.append({
        "id": row.id,
        "y_true": int(row.y_true),
        "fold": int(row.fold),
        **features,
    })

    if i % 30 == 0:
        print(f"Extracted {i}/{len(df)} DEMs")

terrain_df = pd.DataFrame(records)

assert len(terrain_df) == 180
assert terrain_df["id"].is_unique

terrain_df.to_csv(
    OUT_DIR / "terrain_features.csv",
    index=False
)

print("Feature extraction complete.")
display(terrain_df.head())

Extracted 30/180 DEMs
Extracted 60/180 DEMs
Extracted 90/180 DEMs
Extracted 120/180 DEMs
Extracted 150/180 DEMs
Extracted 180/180 DEMs
Feature extraction complete.


,id,y_true,fold,elev_std_m,elev_range_m,elev_iqr_m,elev_p10_relative_m,elev_p90_relative_m,gradient_mean,gradient_std,gradient_p90,roughness_x_m,roughness_y_m
0,0014930068,2,1,1.971896,8.069580,3.744385,-2.208203,2.896899,0.018274,0.014931,0.035939,0.109160,0.099022
1,0014997588,2,1,1.171540,8.938725,1.750136,-1.580046,1.257637,0.027688,0.034740,0.059536,0.153142,0.167914
2,0016282459,1,1,4.051146,17.585603,3.711294,-8.387176,3.897125,0.066851,0.076938,0.182953,0.401750,0.359057
3,0031800424,1,1,1.254297,17.611893,1.335171,-1.792014,1.146895,0.026410,0.031441,0.051600,0.136613,0.165833
4,0032560290,2,1,1.027924,7.478783,0.880430,-1.032600,0.611737,0.026315,0.030606,0.064209,0.115204,0.165520


**Check feature quality**

List the feature columns and count missing or infinite values, then show a numeric summary. The assert at the end refuses to train on a non-finite table.

In [11]:
FEATURE_COLS = [
    col for col in terrain_df.columns
    if col not in ["id", "y_true", "fold"]
]

print("Features:", FEATURE_COLS)

print("\nMissing values:")
print(terrain_df[FEATURE_COLS].isna().sum())

print("\nInfinite values:")
print(
    np.isinf(
        terrain_df[FEATURE_COLS].to_numpy()
    ).sum()
)

print("\nFeature summary:")
display(terrain_df[FEATURE_COLS].describe().T)

assert np.isfinite(
    terrain_df[FEATURE_COLS].to_numpy()
).all()

Features: ['elev_std_m', 'elev_range_m', 'elev_iqr_m', 'elev_p10_relative_m', 'elev_p90_relative_m', 'gradient_mean', 'gradient_std', 'gradient_p90', 'roughness_x_m', 'roughness_y_m']

Missing values:
elev_std_m             0
elev_range_m           0
elev_iqr_m             0
elev_p10_relative_m    0
elev_p90_relative_m    0
gradient_mean          0
gradient_std           0
gradient_p90           0
roughness_x_m          0
roughness_y_m          0
dtype: int64

Infinite values:
0

Feature summary:


,count,mean,std,min,25%,50%,75%,max
elev_std_m,180.0,3.656265,2.780484,0.271703,1.395995,2.963188,4.823627,12.919339
elev_range_m,180.0,18.604779,13.788530,2.212128,8.651283,14.652687,24.661275,79.102570
elev_iqr_m,180.0,4.792713,3.984053,0.338501,1.644876,3.909645,6.472044,19.264671
elev_p10_relative_m,180.0,-4.624380,3.792326,-18.473264,-6.360552,-3.626427,-1.599583,-0.283173
elev_p90_relative_m,180.0,4.711447,4.535402,0.274605,1.442031,3.506663,6.248672,27.030095
gradient_mean,180.0,0.048501,0.028654,0.009405,0.027661,0.041436,0.060118,0.136784
gradient_std,180.0,0.049737,0.028847,0.011457,0.027970,0.042884,0.065031,0.143252
gradient_p90,180.0,0.112295,0.071960,0.018098,0.058780,0.089237,0.145386,0.328516
roughness_x_m,180.0,0.244509,0.153463,0.039319,0.134338,0.201972,0.316615,0.862598
roughness_y_m,180.0,0.310556,0.199245,0.068905,0.166595,0.254123,0.371007,1.073993


**Define lightweight models**

Set up a balanced logistic regression and a shallow random forest. Both impute inside the pipeline so held-out folds do not affect preprocessing.

In [12]:
models = {
    "LogisticRegression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42,
        )),
    ]),

    "RandomForest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            max_depth=4,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        )),
    ]),
}

**Train and evaluate on the five saved folds**

Fit each model on four geographic folds and predict the held-out fold. Out-of-fold predictions and per-fold metrics are what the CNN comparison uses.

In [13]:
prediction_records = []
fold_metric_records = []

X = terrain_df[FEATURE_COLS]
y = terrain_df["y_true"]

for model_name, model_template in models.items():
    print(f"\n=== {model_name} ===")

    for fold in sorted(terrain_df["fold"].unique()):
        train_mask = terrain_df["fold"] != fold
        test_mask = terrain_df["fold"] == fold

        model = clone(model_template)

        model.fit(
            X.loc[train_mask],
            y.loc[train_mask],
        )

        preds = model.predict(X.loc[test_mask])
        probs = model.predict_proba(X.loc[test_mask])

        test_rows = terrain_df.loc[
            test_mask, ["id", "y_true", "fold"]
        ].reset_index(drop=True)

        class_order = model.named_steps[
            "classifier"
        ].classes_

        prob_df = pd.DataFrame(
            probs,
            columns=[f"prob_{c}" for c in class_order],
        )

        for class_id in [0, 1, 2]:
            if f"prob_{class_id}" not in prob_df:
                prob_df[f"prob_{class_id}"] = 0.0

        test_rows["y_pred"] = preds
        test_rows["model"] = model_name

        for class_id in [0, 1, 2]:
            test_rows[f"prob_{class_id}"] = (
                prob_df[f"prob_{class_id}"].to_numpy()
            )

        prediction_records.append(test_rows)

        fold_metrics = {
            "model": model_name,
            "fold": int(fold),
            "n_test": int(test_mask.sum()),
            "accuracy": accuracy_score(
                y.loc[test_mask], preds
            ),
            "balanced_accuracy": balanced_accuracy_score(
                y.loc[test_mask], preds
            ),
            "macro_f1": f1_score(
                y.loc[test_mask],
                preds,
                labels=[0, 1, 2],
                average="macro",
                zero_division=0,
            ),
        }

        fold_metric_records.append(fold_metrics)

        print(
            f"Fold {fold}: "
            f"macro-F1={fold_metrics['macro_f1']:.4f}, "
            f"balanced_acc={fold_metrics['balanced_accuracy']:.4f}"
        )

handcrafted_oof = pd.concat(
    prediction_records,
    ignore_index=True
)

handcrafted_fold_metrics = pd.DataFrame(
    fold_metric_records
)

handcrafted_oof.to_csv(
    OUT_DIR / "oof_predictions.csv",
    index=False
)

handcrafted_fold_metrics.to_csv(
    OUT_DIR / "fold_metrics.csv",
    index=False
)

print("\nSaved handcrafted predictions and fold metrics.")


=== LogisticRegression ===
Fold 1: macro-F1=0.2401, balanced_acc=0.2667
Fold 2: macro-F1=0.2116, balanced_acc=0.4533
Fold 3: macro-F1=0.2167, balanced_acc=0.1867
Fold 4: macro-F1=0.2357, balanced_acc=0.2111
Fold 5: macro-F1=0.3433, balanced_acc=0.3528

=== RandomForest ===
Fold 1: macro-F1=0.2927, balanced_acc=0.2933
Fold 2: macro-F1=0.3822, balanced_acc=0.5733
Fold 3: macro-F1=0.2444, balanced_acc=0.2267
Fold 4: macro-F1=0.2593, balanced_acc=0.2778
Fold 5: macro-F1=0.2320, balanced_acc=0.2444

Saved handcrafted predictions and fold metrics.


**Validate out-of-fold predictions**

Check that each model scored every facility once, on the same fold and label as the CNN file. Probability columns must be finite and sum to one.

In [14]:
for model_name in models:
    subset = handcrafted_oof[
        handcrafted_oof["model"] == model_name
    ]

    assert len(subset) == 180
    assert subset["id"].is_unique
    assert set(subset["id"]) == set(cnn_oof["id"])

    merged = subset.merge(
        cnn_oof[["id", "fold", "y_true"]],
        on="id",
        suffixes=("_handcrafted", "_cnn"),
        validate="one_to_one",
    )

    assert (
        merged["fold_handcrafted"]
        == merged["fold_cnn"]
    ).all()

    assert (
        merged["y_true_handcrafted"]
        == merged["y_true_cnn"]
    ).all()

    p = subset[["prob_0", "prob_1", "prob_2"]].to_numpy()

    assert np.isfinite(p).all()
    assert np.allclose(p.sum(axis=1), 1.0, atol=1e-5)

    print(f"{model_name}: all 180 predictions verified.")

LogisticRegression: all 180 predictions verified.
RandomForest: all 180 predictions verified.


**Compare against the CNN**

Score the handcrafted models, the CNN, and a majority-class baseline with accuracy, balanced accuracy, macro-F1, and per-class recall. This is the direct read on whether summary terrain features match the CNN.

In [15]:
comparison_rows = []

def add_comparison(name, true, pred):
    comparison_rows.append({
        "model": name,
        "accuracy": accuracy_score(true, pred),
        "balanced_accuracy": balanced_accuracy_score(true, pred),
        "macro_f1": f1_score(
            true,
            pred,
            labels=[0, 1, 2],
            average="macro",
            zero_division=0,
        ),
        "small_recall": (
            (pred[true == 0] == 0).mean()
        ),
        "medium_recall": (
            (pred[true == 1] == 1).mean()
        ),
        "large_recall": (
            (pred[true == 2] == 2).mean()
        ),
    })

cnn_true = cnn_oof["y_true"].to_numpy()
cnn_pred = cnn_oof["y_pred"].to_numpy()

add_comparison(
    "Majority baseline",
    cnn_true,
    np.full_like(cnn_true, 2),
)

add_comparison(
    "USGS CNN",
    cnn_true,
    cnn_pred,
)

for model_name in models:
    subset = handcrafted_oof[
        handcrafted_oof["model"] == model_name
    ]

    add_comparison(
        model_name,
        subset["y_true"].to_numpy(),
        subset["y_pred"].to_numpy(),
    )

comparison = pd.DataFrame(comparison_rows)

comparison.to_csv(
    OUT_DIR / "model_comparison.csv",
    index=False
)

display(
    comparison.sort_values(
        "macro_f1",
        ascending=False
    ).round(4)
)

,model,accuracy,balanced_accuracy,macro_f1,small_recall,medium_recall,large_recall
3,RandomForest,0.4000,0.3522,0.3060,0.2857,0.34,0.4309
1,USGS CNN,0.6333,0.3208,0.2881,0.0000,0.06,0.9024
0,Majority baseline,0.6833,0.3333,0.2706,0.0000,0.00,1.0000
2,LogisticRegression,0.3222,0.3222,0.2646,0.2857,0.38,0.3008


**Classification reports and confusion matrices**

Print a per-class report and confusion matrix for each handcrafted model. The matrices show which size classes the simple features confuse.

In [16]:
for model_name in models:
    subset = handcrafted_oof[
        handcrafted_oof["model"] == model_name
    ]

    print(f"\n{'=' * 50}")
    print(model_name)
    print("=" * 50)

    print(classification_report(
        subset["y_true"],
        subset["y_pred"],
        labels=[0, 1, 2],
        target_names=["Small", "Medium", "Large"],
        zero_division=0,
    ))

    matrix = confusion_matrix(
        subset["y_true"],
        subset["y_pred"],
        labels=[0, 1, 2],
    )

    display(pd.DataFrame(
        matrix,
        index=["Actual Small", "Actual Medium", "Actual Large"],
        columns=["Pred Small", "Pred Medium", "Pred Large"],
    ))


LogisticRegression
              precision    recall  f1-score   support

       Small       0.04      0.29      0.07         7
      Medium       0.26      0.38      0.31        50
       Large       0.67      0.30      0.42       123

    accuracy                           0.32       180
   macro avg       0.32      0.32      0.26       180
weighted avg       0.53      0.32      0.37       180



,Pred Small,Pred Medium,Pred Large
Actual Small,2,2,3
Actual Medium,16,19,15
Actual Large,35,51,37



RandomForest
              precision    recall  f1-score   support

       Small       0.08      0.29      0.12         7
      Medium       0.24      0.34      0.28        50
       Large       0.63      0.43      0.51       123

    accuracy                           0.40       180
   macro avg       0.32      0.35      0.31       180
weighted avg       0.50      0.40      0.43       180



,Pred Small,Pred Medium,Pred Large
Actual Small,2,1,4
Actual Medium,6,17,27
Actual Large,17,53,53


**Save experiment metadata**

Write the cohort size, feature list, model settings, and CNN comparison path next to the prediction files so the run can be described without rerunning it.

In [17]:
metadata = {
    "experiment": "usgs_handcrafted_spatial_v1",
    "n_facilities": int(len(terrain_df)),
    "n_folds": int(terrain_df["fold"].nunique()),
    "class_mapping": {
        "0": "Small (<20,000 sqft)",
        "1": "Medium (20,000–99,999 sqft)",
        "2": "Large (>=100,000 sqft)",
    },
    "features": FEATURE_COLS,
    "models": {
        "LogisticRegression": {
            "class_weight": "balanced",
            "max_iter": 2000,
        },
        "RandomForest": {
            "n_estimators": 200,
            "max_depth": 4,
            "min_samples_leaf": 5,
            "class_weight": "balanced",
        },
    },
    "evaluation": (
        "Five-fold geographic-block-disjoint "
        "out-of-fold evaluation using saved USGS folds"
    ),
    "cnn_comparison_source": str(
        CNN_DIR / "oof_predictions.csv"
    ),
    "limitations": [
        "Only 180 facilities",
        "Seven Small-class examples",
        "No independent untouched test cohort",
        "Exploratory model comparisons",
        "NAIP alignment footprints not independently verified",
    ],
}

with open(OUT_DIR / "experiment_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved all handcrafted baseline artifacts to:")
print(OUT_DIR)

print("\nFiles:")
for path in sorted(OUT_DIR.iterdir()):
    print("-", path.name)

Saved all handcrafted baseline artifacts to:
/mnt/c/Users/attho/OneDrive/Desktop/College/ACM ATLAS/ATLAS/outputs/usgs_handcrafted_spatial_v1

Files:
- experiment_metadata.json
- fold_metrics.csv
- model_comparison.csv
- oof_predictions.csv
- terrain_features.csv
